<a href="https://colab.research.google.com/github/kiryu-arai/kaggle_compedition_monster/blob/suzuki/v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# カリフォルニア住宅価格予測 - 発展的な特徴量エンジニアリングと交差検証パイプライン

このノートブックでは、以下の5つのアプローチを統合した高度なパイプラインを実行します。
1. **PCA（主成分分析）** によるサイズ関連変数の次元縮約
2. **主要都市（SF/LA）からの距離** の算出による地理情報の集約
3. **K-Meansクラスタリング** による地域ブロックの自動グループ化
4. **5-Fold 交差検証（Cross Validation）** による頑健な評価とアンサンブル予測
5. **目的変数（Price）の対数変換（log1p/expm1）** による予測精度の安定化

In [31]:
# kaggle APIのインストール
!pip install kaggle

In [32]:
# driveのマウント
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
# パイプライン構成
import os
import json

# カレントディレクトリ、または適切なパスから kaggle.json を読み込んでください
with open("kaggle.json", 'r') as f:
    json_data = json.load(f)
os.environ['KAGGLE_USERNAME'] = json_data['username']
os.environ['KAGGLE_KEY'] = json_data['key']

In [34]:
# データのダウンロードと解凍
!kaggle competitions download -c ambl-california-housing
!unzip -o /content/ambl-california-housing.zip

ambl-california-housing.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/ambl-california-housing.zip
  inflating: sample.csv              
  inflating: test.csv                
  inflating: train.csv               


In [35]:
# データの移動
import os
import shutil

destination_folder = '/content/drive/MyDrive/kaggle_data'

if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)
    print(f"フォルダ '{destination_folder}' を作成しました。")
else:
    print(f"フォルダ '{destination_folder}' は既に存在します。")

files_to_move = ['sample.csv', 'test.csv', 'train.csv']

for file_name in files_to_move:
    source_path = os.path.join('/content/', file_name)
    destination_path = os.path.join(destination_folder, file_name)
    if os.path.exists(source_path):
        shutil.move(source_path, destination_path)
        print(f"'{file_name}' を '{destination_folder}' に移動しました。")
    else:
        print(f"'{file_name}' は存在しませんでした。")

フォルダ '/content/drive/MyDrive/kaggle_data' は既に存在します。
'sample.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'test.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'train.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。


### モジュールの準備

In [36]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

%matplotlib inline

### データセットの読み込み

In [37]:
train = pd.read_csv('/content/drive/MyDrive/kaggle_data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/kaggle_data/test.csv')
sample = pd.read_csv('/content/drive/MyDrive/kaggle_data/sample.csv')

### 特徴量エンジニアリング（FE）の定義と実行

In [48]:
def advanced_feature_engineering_v2(train_df, test_df):
    # 訓練データとテストデータを結合して一括で処理
    df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # 既存のベース特徴量
    df['Household'] = df['Population'] / df['AveOccup']
    df['AllRooms'] = df['Household'] * df['AveRooms']
    df['AllBedrms'] = df['Household'] * df['AveBedrms']
    df['RoomsPerBedroom'] = df['AveRooms'] / df['AveBedrms']
    df['BedroomRatio'] = df['AveBedrms'] / df['AveRooms']

    # 【新規追加】部屋数・寝室数関連の追加特徴量
    # 1. 平均部屋数と平均寝室数の差分: 共用スペースの多さを示す
    df['RoomsMinusBedrms'] = df['AveRooms'] - df['AveBedrms']
    # 2. 総部屋数／人口: 一人当たりの総部屋数 (Household * AveRooms / Population と同じだが、明示的に定義)
    df['AllRoomsPerCapita'] = df['AllRooms'] / df['Population']
    # 3. 総寝室数／人口: 一人当たりの総寝室数
    df['AllBedrmsPerCapita'] = df['AllBedrms'] / df['Population']
    # 4. 【新規追加】一人当たりの部屋数: 総部屋数 / 人口
    df['RoomsPerPerson'] = df['AllRooms'] / df['Population']


    # 緯度と経度を集約 (SF・LAの中心部からのユークリッド距離)
    sf_coord = (37.7749, -122.4194)
    la_coord = (34.0522, -118.2437)
    df['dist_to_SF'] = np.sqrt((df['Latitude'] - sf_coord[0])**2 + (df['Longitude'] - sf_coord[1])**2)
    df['dist_to_LA'] = np.sqrt((df['Latitude'] - la_coord[0])**2 + (df['Longitude'] - la_coord[1])**2)

    # 【Step 2】地理ドメイン知識の追加 (サンディエゴへの距離 & 海岸線アプローチ)
    sd_coord = (32.7157, -117.1611)
    df['dist_to_SD'] = np.sqrt((df['Latitude'] - sd_coord[0])**2 + (df['Longitude'] - sd_coord[1])**2)
    # カリフォルニアは西（経度がマイナス側）に行くほど海に近づくため、擬似的な海岸線指標とする
    df['proximity_to_coast'] = df['Longitude']

    # 【新規追加】追加の地理ベース特徴量
    df['Lat_Long'] = df['Latitude'] * df['Longitude']
    df['IsNorth'] = (df['Latitude'] > 36).astype(int)
    df['Coastal'] = (df['Longitude'] > -122).astype(int)


    # K-Meansによる地理的クラスタリング (10地域に分割)
    kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
    df['geo_cluster'] = kmeans.fit_predict(df[['Latitude', 'Longitude']])

    # 【Step 1】エリア集計特徴量 (GroupBy) の追加
    # 各地理クラスタにおける所得・家屋年齢の平均値を算出
    cluster_medinc_mean = df.groupby('geo_cluster')['MedInc'].transform('mean')
    cluster_age_mean = df.groupby('geo_cluster')['HouseAge'].transform('mean')

    # 「そのブロックが、周囲の地域平均と比べてどうなのか」という相対的な差分を計算
    df['diff_MedInc_from_cluster'] = df['MedInc'] - cluster_medinc_mean
    df['diff_HouseAge_from_cluster'] = df['HouseAge'] - cluster_age_mean

    # PCA (主成分分析) による次元縮約
    pca_cols = ['AveRooms', 'AveBedrms', 'Population', 'Household', 'AllRooms', 'AllBedrms']
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(df[pca_cols])

    pca = PCA(n_components=2, random_state=42)
    pca_results = pca.fit_transform(scaled_features)
    df['pca_dim1'] = pca_results[:, 0]
    df['pca_dim2'] = pca_results[:, 1]

    # 多項式・交差特徴量の追加
    df['MedInc_sq'] = df['MedInc']**2
    df['HouseAge_sq'] = df['HouseAge']**2
    df['Lat_sq'] = df['Latitude']**2
    df['Lat_Lon_interaction'] = df['Latitude'] * df['Longitude']

    # 処理後に再び訓練データとテストデータに再分割
    train_fe = df[df['Price'].notnull()].copy()
    test_fe = df[df['Price'].isnull()].copy().drop(['Price'], axis=1)

    return train_fe, test_fe

print("強化版特徴量エンジニアリングを実行中...")
train_fe, test_fe = advanced_feature_engineering_v2(train, test)
print("データ加工が完了しました！")

強化版特徴量エンジニアリングを実行中...
データ加工が完了しました！


In [49]:
# 1. Priceとの相関係数を計算し、.abs()で全数値を絶対値（プラス）に変換
# 2. 絶対値が大きい順（降順）にソートする
price_corr_abs = train_fe.corr()[['Price']].abs().sort_values(by='Price', ascending=False)

# 3. カラム名を「Price_Abs（絶対値）」に変更してわかりやすくする
price_corr_abs.columns = ['Price_Abs']
# 4. 結果の表示
price_corr_abs

,Price_Abs
Price,1.000000
MedInc,0.689659
MedInc_sq,0.626316
diff_MedInc_from_cluster,0.590998
RoomsPerBedroom,0.387964
BedroomRatio,0.259745
AllRoomsPerCapita,0.210773
RoomsPerPerson,0.210773
RoomsMinusBedrms,0.200562
Coastal,0.161846


### 5-Fold 交差検証と対数変換を用いたモデルの学習・評価

In [50]:
from xgboost import XGBRegressor

# 説明変数と目的変数の分離
features = [c for c in train_fe.columns if c not in ['Price', 'id']]
X = train_fe[features]
y = train_fe['Price']
X_test = test_fe[features]

# 5-Fold 交差検証の準備
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# それぞれのモデル用のOOF（検証データ予測）格納配列
oof_lgb = np.zeros(len(train_fe))
oof_xgb = np.zeros(len(train_fe))

# テストデータ予測用の格納配列
preds_lgb = np.zeros(len(test_fe))
preds_xgb = np.zeros(len(test_fe))

# 【Step 4】Optunaでの探索結果を反映した、最適化済みのパラメータ群
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'random_state': 42,
    'n_estimators': 2500,
    'learning_rate': 0.02,
    'max_depth': 6,
    'num_leaves': 45,           # 表現力を少し強化
    'subsample': 0.8,
    'colsample_bytree': 0.7,    # 特徴量を少し間引いて過学習を防ぐ
    'reg_alpha': 0.5,           # L1正則化
    'reg_lambda': 1.0,          # L2正則化
    'verbosity': -1
}

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'random_state': 42,
    'n_estimators': 2500,
    'learning_rate': 0.02,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'n_jobs': -1
}

print("【マルチモデル × 5-Fold CV】の同時学習を開始します...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # 目的変数（Price）の対数変換
    y_train_log = np.log1p(y_train)
    y_val_log = np.log1p(y_val)

    # --- 1. LightGBM の学習 ---
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_train, y_train_log,
        eval_set=[(X_val, y_val_log)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    oof_lgb[val_idx] = np.expm1(model_lgb.predict(X_val))
    preds_lgb += np.expm1(model_lgb.predict(X_test)) / kf.n_splits

    # --- 2. XGBoost の学習 ---
    model_xgb = XGBRegressor(**xgb_params)
    model_xgb.fit(
        X_train, y_train_log,
        eval_set=[(X_val, y_val_log)],
        verbose=False
        # ※ XGBoostのearly_stoppingは内部で自動処理、またはxgb_params等で制御可能ですが、
        # ここでは学習率と学習回数をLGBMと揃えて安定させています。
    )
    oof_xgb[val_idx] = np.expm1(model_xgb.predict(X_val))
    preds_xgb += np.expm1(model_xgb.predict(X_test)) / kf.n_splits

    print(f"-> Fold {fold+1} が完了しました。")

# 各単体モデルのCVスコア算出
lgb_cv = np.sqrt(mean_squared_error(y, np.clip(oof_lgb, 0, 5.00001)))
xgb_cv = np.sqrt(mean_squared_error(y, np.clip(oof_xgb, 0, 5.00001)))
print(f"\n[単体スコア] LightGBM CV: {lgb_cv:.5f} | XGBoost CV: {xgb_cv:.5f}")

# 【Step 3】予測値のブレンディング (LightGBM: 60%, XGBoost: 40%)
final_oof = (oof_lgb * 0.6) + (oof_xgb * 0.4)
final_oof = np.clip(final_oof, 0, 5.00001)

final_test_preds = (preds_lgb * 0.6) + (preds_xgb * 0.4)
final_test_preds = np.clip(final_test_preds, 0, 5.00001)

# ブレンド後の総合CVスコア
blend_cv = np.sqrt(mean_squared_error(y, final_oof))
print(f"★ 【ブレンド後】総合交差検証（CV）RMSE: {blend_cv:.5f}")

# %% [markdown]
# ### 提出用ファイルの作成と保存
sample_cv_advanced = sample.copy()
sample_cv_advanced['Price'] = final_test_preds
output_path = '/content/drive/MyDrive/kaggle_data/submit_final_blend.csv'
sample_cv_advanced.to_csv(output_path, index=None)

print(f"最終提出用ファイル '{output_path}' を作成しました。Kaggleへの提出準備は万端です！")

【マルチモデル × 5-Fold CV】の同時学習を開始します...
-> Fold 1 が完了しました。
-> Fold 2 が完了しました。
-> Fold 3 が完了しました。
-> Fold 4 が完了しました。
-> Fold 5 が完了しました。

[単体スコア] LightGBM CV: 0.44130 | XGBoost CV: 0.43913
★ 【ブレンド後】総合交差検証（CV）RMSE: 0.43848
最終提出用ファイル '/content/drive/MyDrive/kaggle_data/submit_final_blend.csv' を作成しました。Kaggleへの提出準備は万端です！


### 提出用ファイルの作成と保存

In [51]:
sample_cv_advanced = sample.copy()
sample_cv_advanced['Price'] = final_test_preds
output_path = '/content/drive/MyDrive/kaggle_data/submit_final_blend.csv'
sample_cv_advanced.to_csv(output_path, index=None)

print(f"最終提出用ファイル '{output_path}' を作成しました。Kaggleへの提出準備は万端です！")

最終提出用ファイル '/content/drive/MyDrive/kaggle_data/submit_final_blend.csv' を作成しました。Kaggleへの提出準備は万端です！


### Kaggleへの直接投稿

In [52]:
# 作成したファイルをKaggleに直接投稿
!kaggle competitions submit -c ambl-california-housing -f /content/drive/MyDrive/kaggle_data/submit_final_blend.csv -m "Advanced FE and 5-Fold CV Submission via Colab"

100% 93.7k/93.7k [00:01<00:00, 90.1kB/s]
Successfully submitted to AMBL初心者向けコンペティション_California Housing